In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


In [13]:
import zipfile
import os

# Define the path to the input directory
input_path = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'

# List of zip files to extract
zip_files = ['train.csv.zip', 'features.csv.zip', 'test.csv.zip']

for zip_file in zip_files:
    zip_path = os.path.join(input_path, zip_file)
    # Extract to the same directory (or a specific output dir like '/kaggle/working/')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/kaggle/working/')
        
print("Files extracted successfully!")


Files extracted successfully!


In [14]:
import pandas as pd

# 1. Load Data (assuming you extracted them to /kaggle/working/)
train = pd.read_csv('/kaggle/working/train.csv')
features = pd.read_csv('/kaggle/working/features.csv')
stores = pd.read_csv('/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv')
test = pd.read_csv('/kaggle/working/test.csv')

# 2. Convert Date to datetime
train['Date'] = pd.to_datetime(train['Date'])
features['Date'] = pd.to_datetime(features['Date'])
test['Date'] = pd.to_datetime(test['Date'])

# 3. Merge Data
# First merge train with features on Store and Date
df_train = train.merge(features, on=['Store', 'Date'], how='left')
# Then merge with stores on Store ID
df_train = df_train.merge(stores, on='Store', how='left')

# Do the same for test set so it has all features available during inference
df_test = test.merge(features, on=['Store', 'Date'], how='left')
df_test = df_test.merge(stores, on='Store', how='left')

# 4. Chronological Split
# Define the cutoff date for Validation
VAL_START_DATE = '2012-09-01'

# Filter Train and Val sets from the merged training data
df_val = df_train[df_train['Date'] >= VAL_START_DATE].copy()
df_train_final = df_train[df_train['Date'] < VAL_START_DATE].copy()

print(f"Training shape: {df_train_final.shape}")
print(f"Validation shape: {df_val.shape}")
print(f"Test shape: {df_test.shape}")

# Optional: Drop the original 'Date' column if you've already extracted Year/Month/etc., 
# but keep it for now as some models (like Prophet or PyTorch Forecasting) need it.


Training shape: (397841, 17)
Validation shape: (23729, 17)
Test shape: (115064, 16)


In [15]:
import pandas as pd

# Assuming df_train_final and df_val are already merged and sorted by Date
# It is CRITICAL that the dataframe is sorted by Date before calculating lags/rolling stats

def create_temporal_features(df):
    """
    Adds lag features and rolling statistics for Weekly_Sales 
    grouped by Store and Dept.
    """
    # Sort by Store, Dept, and Date to ensure correct ordering within groups
    df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    
    # Define the groups for grouping operations
    group_cols = ['Store', 'Dept']
    
    # 1. Lag Features: Sales from previous weeks
    # Lag 1: Last week's sales
    df['sales_lag_1'] = df.groupby(group_cols)['Weekly_Sales'].shift(1)
    
    # Lag 2: Two weeks ago
    df['sales_lag_2'] = df.groupby(group_cols)['Weekly_Sales'].shift(2)
    
    # Lag 3: Three weeks ago
    df['sales_lag_3'] = df.groupby(group_cols)['Weekly_Sales'].shift(3)
    
    # Lag 4: Four weeks ago (useful for monthly trends)
    df['sales_lag_4'] = df.groupby(group_cols)['Weekly_Sales'].shift(4)
    
    # Lag 52: Same week last year (captures yearly seasonality)
    df['sales_lag_52'] = df.groupby(group_cols)['Weekly_Sales'].shift(52)
    
    # 2. Rolling Statistics: Average of last N weeks
    # Rolling Mean of last 4 weeks
    df['roll_mean_4'] = df.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
    )
    
    # Rolling Std Dev of last 4 weeks (captures volatility)
    df['roll_std_4'] = df.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=4, min_periods=1).std()
    )
    
    # Rolling Mean of last 8 weeks (longer trend)
    df['roll_mean_8'] = df.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=8, min_periods=1).mean()
    )
    
    return df

# Apply to training set
df_train_final = create_temporal_features(df_train_final)

# IMPORTANT: For validation/test sets, you must calculate these features 
# using the history from the training set to avoid leakage.
# The easiest way in a notebook workflow is to concatenate train + val + test,
# sort by date, calculate features, then split again.
# HOWEVER, since we already split chronologically, we can calculate them on the combined set
# but only use the calculated values for the respective splits.

# Better approach for strict no-leakage:
# Combine Train and Val temporarily to calculate lags that span the boundary
df_combined = pd.concat([df_train_final, df_val], ignore_index=True)
df_combined = create_temporal_features(df_combined)

# Split back out
df_train_final = df_combined[df_combined['Date'] < '2012-09-01'].copy()
df_val = df_combined[df_combined['Date'] >= '2012-09-01'].copy()

# Do the same for Test Set if needed for inference later
# Note: For test set, you'll need to append it to the end of the historical data 
# to calculate its initial lags correctly during inference.

print("Temporal features added.")
print(df_train_final[['Date', 'sales_lag_1', 'roll_mean_4']].head())


Temporal features added.
        Date  sales_lag_1   roll_mean_4
0 2010-02-05          NaN           NaN
1 2010-02-12     24924.50  24924.500000
2 2010-02-19     46039.49  35481.995000
3 2010-02-26     41595.55  37519.846667
4 2010-03-05     19403.54  32990.770000


In [16]:
!pip install wandb -q

import wandb
import os

# Retrieve the secret from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: WANDB_API_KEY not found in secrets.")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [23]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import wandb
import os
import joblib

# ---------------------------------------------------------
# 0. INITIALIZE WANDB
# ---------------------------------------------------------
api_key = user_secrets.get_secret("WANDB_API_KEY")

if api_key:
    run = wandb.init(
        project="walmart-sales-forecasting",
        name="XGBoost_Baseline_WandB_Only",
        config={
            "learning_rate": 0.05,
            "max_depth": 6,
            "n_estimators": 500,
            "objective": "reg:squarederror",
            "seed": 42
        }
    )
else:
    print("Warning: WANDB_API_KEY not found.")
    run = None

# ---------------------------------------------------------
# 1. FIX MERGE ISSUES & PREPROCESSING
# ---------------------------------------------------------

# Fix the duplicate IsHoliday columns from the merge
for df_name, df_obj in [('df_train_final', df_train_final), ('df_val', df_val)]:
    if 'IsHoliday_x' in df_obj.columns and 'IsHoliday' not in df_obj.columns:
        df_obj.rename(columns={'IsHoliday_x': 'IsHoliday'}, inplace=True)
        if 'IsHoliday_y' in df_obj.columns:
            df_obj.drop(columns=['IsHoliday_y'], inplace=True)
    elif 'IsHoliday_y' in df_obj.columns and 'IsHoliday' not in df_obj.columns:
        df_obj.rename(columns={'IsHoliday_y': 'IsHoliday'}, inplace=True)
        if 'IsHoliday_x' in df_obj.columns:
            df_obj.drop(columns=['IsHoliday_x'], inplace=True)

def preprocess_for_xgb(df):
    df_clean = df.copy()
    df_clean.columns = df_clean.columns.str.strip()
    
    markdown_cols = [c for c in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5'] if c in df_clean.columns]
    df_clean[markdown_cols] = df_clean[markdown_cols].fillna(0)
    
    type_map = {'A': 1, 'B': 2, 'C': 3}
    if 'Type' in df_clean.columns:
        df_clean['Type_encoded'] = df_clean['Type'].map(type_map).fillna(0).astype(int)
    else:
        df_clean['Type_encoded'] = 0
        
    if 'IsHoliday' in df_clean.columns:
        df_clean['IsHoliday_int'] = df_clean['IsHoliday'].astype(int)
    else:
        df_clean['IsHoliday_int'] = 0
        
    if 'Date' in df_clean.columns:
        df_clean['Year'] = df_clean['Date'].dt.year
        df_clean['Month'] = df_clean['Date'].dt.month
        df_clean['DayOfWeek'] = df_clean['Date'].dt.dayofweek 
        df_clean['WeekOfYear'] = df_clean['Date'].dt.isocalendar().week.astype(int)
    else:
        df_clean['Year'] = 2012
        df_clean['Month'] = 1
        df_clean['DayOfWeek'] = 0
        df_clean['WeekOfYear'] = 1
    
    cols_to_drop = ['Store', 'Dept', 'Id', 'Date', 'Type', 'IsHoliday', 'Weekly_Sales']
    cols_to_drop = [col for col in cols_to_drop if col in df_clean.columns]
    
    return df_clean.drop(columns=cols_to_drop)

# ---------------------------------------------------------
# 2. APPLY TO TRAIN AND VAL SETS
# ---------------------------------------------------------

X_train = preprocess_for_xgb(df_train_final)
y_train = df_train_final['Weekly_Sales'] 

X_val = preprocess_for_xgb(df_val)
y_val = df_val['Weekly_Sales']

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")

object_cols = X_train.select_dtypes(include=['object']).columns
if len(object_cols) > 0:
    print(f"One-hot encoding columns: {list(object_cols)}")
    X_train = pd.get_dummies(X_train, columns=object_cols)
    X_val = pd.get_dummies(X_val, columns=object_cols)

train_cols = X_train.columns
val_cols = X_val.columns
missing_in_val = set(train_cols) - set(val_cols)
extra_in_val = set(val_cols) - set(train_cols)

for col in missing_in_val:
    X_val[col] = 0 
for col in extra_in_val:
    X_val = X_val.drop(columns=[col])

X_val = X_val.reindex(columns=train_cols)


# ---------------------------------------------------------
# 3. TRAIN XGBOOST MODEL
# ---------------------------------------------------------

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    'learning_rate': 0.05,
    'max_depth': 6,
    'n_estimators': 500,
    'early_stopping_rounds': 20,
    'seed': 42,
    'verbosity': 1 
}

model_params = {k:v for k,v in params.items() if k != 'early_stopping_rounds'}
model = xgb.XGBRegressor(**model_params)

try:
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False 
    )
except Exception as e:
    print(f"Error during fitting: {e}")


# ---------------------------------------------------------
# 4. FINAL EVALUATION & LOGGING
# ---------------------------------------------------------
y_pred = model.predict(X_val)

if 'IsHoliday' in df_val.columns:
    is_holiday = df_val['IsHoliday'].values
else:
    is_holiday = np.zeros(len(y_val))

weights = np.where(is_holiday, 5, 1)
final_wmae = np.average(np.abs(y_val - y_pred), weights=weights)
final_mae = mean_absolute_error(y_val, y_pred)

if run:
    # Log Final Metrics
    wandb.log({"final_validation_wmae": final_wmae, "final_validation_mae": final_mae})
    
    # --- FIXED FEATURE IMPORTANCE LOGGING ---
    importance = model.feature_importances_
    feature_names = X_train.columns.tolist()
    importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
    importance_df = importance_df.sort_values(by='importance', ascending=False)
    
    # Use wandb.Table which is compatible with all recent versions
    table = wandb.Table(dataframe=importance_df.head(20))
    wandb.log({"feature_importance_table": table})
    
    # Alternatively, you can log a simple bar plot using matplotlib if you prefer visuals
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df.head(20)['feature'], importance_df.head(20)['importance'])
    plt.gca().invert_yaxis()
    plt.title("Top 20 Feature Importances")
    plt.tight_layout()
    wandb.log({"feature_importance_plot": wandb.Image(plt)})
    plt.close()
    # ----------------------------------------
    
    # Save Model Artifact to W&B
    artifact = wandb.Artifact("xgboost-model-final", type="model")
    joblib.dump(model, "model.joblib")
    artifact.add_file("model.joblib")
    run.log_artifact(artifact)

print(f"Final Validation WMAE: {final_wmae:.2f}")
print(f"Final Validation MAE: {final_mae:.2f}")

if run:
    wandb.finish()


final_validation_mae,▁
final_validation_wmae,▁
final_validation_mae,1223.09988
final_validation_wmae,1320.64615


Training features shape: (397841, 24)
Validation features shape: (23729, 24)
Final Validation WMAE: 1320.65
Final Validation MAE: 1223.10


final_validation_mae,▁
final_validation_wmae,▁
final_validation_mae,1223.09988
final_validation_wmae,1320.64615
